# LangGraph email workflow — on a REAL free model (Groq / Mistral)

Same graph as before:

```
START → classify → (route) → spam → drop → END
                            → real → email_agent → END
```

Inside `email_agent` the model reads an email, then wants to reply — and the
graph **pauses for your approval** before it actually sends.

This version runs on a real free model. **Groq** is the default (free, no credit
card). **Mistral** is a one-line swap. Run cells top to bottom.


## 0. Get a free key (one time, ~1 min)

**Groq (recommended):** go to `console.groq.com` → sign in → *API Keys* →
*Create API Key* → copy it. No credit card.

**Mistral (alternative):** go to `console.mistral.ai` → *API Keys* → create one.
Free tier available.


## 1. Install

In [ ]:
!pip install -U langchain langgraph langchain-groq langchain-mistralai -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.0/147.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.9/248.9 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 567.0/567.0 kB 12.9 MB/s eta 0:00:00


## 2. Pick your model
Run the **Groq** block as-is. To use Mistral instead, comment out the Groq lines
and uncomment the Mistral ones. (A no-key mock fallback is at the bottom of the
cell if you ever want to run without a key.)

Model names change over time — if one errors, grab a current tool-calling model
from the provider's console.

In [ ]:
import os, getpass

# ---- Option A: Groq (default) ----
os.environ["GROQ_API_KEY"] = getpass.getpass("Groq API key: ")
from langchain_groq import ChatGroq
model = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

# ---- Option B: Mistral (uncomment to use instead) ----
# os.environ["MISTRAL_API_KEY"] = getpass.getpass("Mistral API key: ")
# from langchain_mistralai import ChatMistralAI
# model = ChatMistralAI(model="mistral-small-latest", temperature=0)

# ---- Option C: no-key mock fallback (uncomment to use instead) ----
# from langchain_core.language_models.chat_models import BaseChatModel
# from langchain_core.messages import AIMessage, ToolMessage
# from langchain_core.outputs import ChatResult, ChatGeneration
# class MockChatModel(BaseChatModel):
#     @property
#     def _llm_type(self): return "mock"
#     def bind_tools(self, tools, **kwargs): return self
#     def _generate(self, messages, stop=None, run_manager=None, **kwargs):
#         done = {m.name for m in messages if isinstance(m, ToolMessage)}
#         if "send_email" in done:
#             msg = AIMessage(content="All done — read the email and sent a reply.")
#         elif "read_email" in done:
#             msg = AIMessage(content="", tool_calls=[{"name":"send_email",
#                 "args":{"to":"boss@corp.com","body":"Here is the Q3 report."},"id":"c2"}])
#         else:
#             msg = AIMessage(content="", tool_calls=[{"name":"read_email","args":{},"id":"c1"}])
#         return ChatResult(generations=[ChatGeneration(message=msg)])
# model = MockChatModel()

print("model ready:", type(model).__name__)

Groq API key: ··········
model ready: ChatGroq


## 3. Sanity check
Confirm the model answers before we build anything.

In [ ]:
import os, requests
r = requests.get("https://api.groq.com/openai/v1/models",
                 headers={"Authorization": f"Bearer {os.environ['GROQ_API_KEY']}"})
print([m["id"] for m in r.json()["data"]])

['openai/gpt-oss-120b', 'allam-2-7b', 'qwen/qwen3.6-27b', 'groq/compound', 'meta-llama/llama-prompt-guard-2-22m', 'groq/compound-mini', 'whisper-large-v3-turbo', 'canopylabs/orpheus-arabic-saudi', 'openai/gpt-oss-20b', 'whisper-large-v3', 'canopylabs/orpheus-v1-english', 'meta-llama/llama-prompt-guard-2-86m', 'openai/gpt-oss-safeguard-20b']


In [ ]:
from langchain_core.messages import HumanMessage
print(model.invoke([HumanMessage(content="Reply with exactly: ready")]).content)

ready


## 4. Tools (the agent's hands)
`read_email` is safe to run unattended. `send_email` is the risky one we gate.

In [ ]:
from langchain_core.tools import tool

@tool
def read_email() -> str:
    """Read the latest incoming email in the inbox."""
    return "From the boss (boss@corp.com): Please send me the Q3 report today."

@tool
def send_email(to: str, body: str) -> str:
    """Send an email. `to` is the recipient address, `body` is the message text."""
    return f"Email sent to {to}."

## 5. The agent, with the human-approval gate
`interrupt_on={"send_email": True}` = pause and ask a human before running
`send_email`. `read_email` isn't listed, so it runs freely.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware

email_agent = create_agent(
    model=model,
    tools=[read_email, send_email],
    middleware=[HumanInTheLoopMiddleware(interrupt_on={"send_email": True})],
)
print("agent built")

agent built


## 6. The outer workflow: state, classify, route
`classify_node` tags the email spam/real (dumb keyword check — this is the
"spam detection defined elsewhere"). `route` returns the next node's name.

In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
from langchain_core.messages import AIMessage

class State(TypedDict):
    messages: Annotated[list, add_messages]
    category: str

def classify_node(state: State):
    text = ""
    for m in state["messages"]:
        if getattr(m, "type", "") == "human":
            text = m.content
    spam_words = ["win", "free money", "lottery", "prince", "click here"]
    cat = "spam" if any(w in text.lower() for w in spam_words) else "real"
    print(f"[classify] -> {cat}")
    return {"category": cat}

def route(state: State):
    return "drop" if state["category"] == "spam" else "email_agent"

def drop_node(state: State):
    print("[drop] spam ignored")
    return {"messages": [AIMessage(content="(ignored as spam)")]}

## 7. Build & compile the graph
`checkpointer=InMemorySaver()` is **required** — it saves state so the run can
pause and resume. Without it, the approval gate can't work.

In [ ]:
from langgraph.graph import START, END, StateGraph
from langgraph.checkpoint.memory import InMemorySaver

graph = (
    StateGraph(State)
    .add_node("classify", classify_node)
    .add_node("email_agent", email_agent)
    .add_node("drop", drop_node)
    .add_edge(START, "classify")
    .add_conditional_edges("classify", route,
                           {"email_agent": "email_agent", "drop": "drop"})
    .add_edge("email_agent", END)
    .add_edge("drop", END)
    .compile(checkpointer=InMemorySaver())
)
print("graph compiled")

graph compiled


## 8. Path A — a spam email
Classified as spam, dropped, never reaches the agent.

In [ ]:
cfg = {"configurable": {"thread_id": "spam-1"}}
out = graph.invoke(
    {"messages": [{"role": "user", "content": "Congratulations, you WIN free money, click here!"}]},
    cfg,
)
print("FINAL:", out["messages"][-1].content)

[classify] -> spam
[drop] spam ignored
FINAL: (ignored as spam)


## 9. Path B — a real email (this PAUSES)
We give the agent a clear instruction so it reads the inbox and then tries to
send a reply — which trips the approval gate. The run returns an `__interrupt__`
instead of finishing.

In [ ]:
cfg = {"configurable": {"thread_id": "real-2"}}
start = {"messages": [
    {"role": "system", "content":
        "You are an email assistant. First call read_email to see the message, "
        "then call send_email to reply appropriately. Always send a reply."},
    {"role": "user", "content": "Please handle my latest email."},
]}
out = graph.invoke(start, cfg)

if "__interrupt__" in out:
    req = out["__interrupt__"][0].value["action_requests"][0]
    print("⏸  PAUSED — approval needed to run:", req["name"])
    print("    to:  ", req["args"].get("to"))
    print("    body:", req["args"].get("body"))
else:
    # A model may occasionally reply without sending; then no pause happens.
    print("No pause — model didn't call send_email. FINAL:", out["messages"][-1].content)

[classify] -> real
⏸  PAUSED — approval needed to run: send_email
    to:   boss@corp.com
    body: Sure, I'll send the Q3 report shortly.


### 9a. Approve → the run resumes and finishes

In [ ]:
# Reject the send:
graph.invoke(Command(resume={"decisions": [{"type": "reject", "message": "Do not send."}]}), cfg)

{'messages': [SystemMessage(content='You are an email assistant. First call read_email to see the message, then call send_email to reply appropriately. Always send a reply.', additional_kwargs={}, response_metadata={}, id='73473089-4662-4ca8-982e-24c4e1b8e849'),
  HumanMessage(content='Please handle my latest email.', additional_kwargs={}, response_metadata={}, id='c1106a86-d739-4412-bc99-84f2a2b71cbe'),
  SystemMessage(content='You are an email assistant. First call read_email to see the message, then call send_email to reply appropriately. Always send a reply.', additional_kwargs={}, response_metadata={}, id='304c572c-a4f9-49cd-aa58-918e110fdbe4'),
  HumanMessage(content='Please handle my latest email.', additional_kwargs={}, response_metadata={}, id='52592da7-4b8b-4a17-bff9-df056ae14641'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to read the latest email. Use read_email function.', 'tool_calls': [{'id': 'fc_110ab492-da58-4322-b29e-eb62a369d29d', 'func

### 9b. Try the other decisions
Rerun cell 9 with a NEW `thread_id` (change `"real-1"` to `"real-2"`), then:

```python
# Reject the send:
graph.invoke(Command(resume={"decisions": [{"type": "reject", "message": "Do not send."}]}), cfg)

# Edit before sending:
graph.invoke(Command(resume={"decisions": [{"type": "edit",
    "edited_action": {"name": "send_email",
        "args": {"to": "boss@corp.com", "body": "Q3 report attached — sent for review."}}}]}), cfg)
```
Allowed decision types: **approve, edit, reject, respond**.

---
### Notes
- If cell 9 doesn't pause, the model chose to answer without sending. Make the
  system instruction firmer, or lower `temperature` (already 0).
- If the model name errors, pick a current tool-calling model from the provider
  console (Groq: e.g. a Llama-3.x "versatile" model; Mistral: `mistral-small-latest`).
- The graph, tools, and gate are model-agnostic — only cell 2 changes to switch
  providers.


In [ ]:
from langgraph.types import Command

cfg = {"configurable": {"thread_id": "reject-clean-1"}}   # fresh, unused id
start = {"messages": [
    {"role": "system", "content":
        "You are an email assistant. Read the email with read_email, then reply with send_email."},
    {"role": "user", "content": "Please handle my latest email."},
]}

out = graph.invoke(start, cfg)                 # 1) should PAUSE
print("paused?", "__interrupt__" in out)

out = graph.invoke(                            # 2) reject it
    Command(resume={"decisions": [{"type": "reject", "message": "Do not send."}]}),
    cfg,
)
print("paused again?", "__interrupt__" in out)
print("final:", out["messages"][-1].content)

[classify] -> real
paused? False
paused again? False
final: I’m ready to send the Q3 report to your boss, but I don’t have the file yet. Could you please upload the Q3 report (PDF, Excel, or whichever format you use) so I can attach it and send the email right away? Once I have the attachment, I’ll take care of it for you.
